In [1]:
library(terra)
library(ncdf4)
library(ggplot2)
library(tidyterra)
library(aws.s3)
library(terra)
library(leaflet)

code for methods in class “Rcpp_SpatCategories” was not checked for suspicious field assignments (recommended package ‘codetools’ not available?)

code for methods in class “Rcpp_SpatCategories” was not checked for suspicious field assignments (recommended package ‘codetools’ not available?)

code for methods in class “Rcpp_SpatDataFrame” was not checked for suspicious field assignments (recommended package ‘codetools’ not available?)

code for methods in class “Rcpp_SpatDataFrame” was not checked for suspicious field assignments (recommended package ‘codetools’ not available?)

code for methods in class “Rcpp_SpatExtent” was not checked for suspicious field assignments (recommended package ‘codetools’ not available?)

code for methods in class “Rcpp_SpatExtent” was not checked for suspicious field assignments (recommended package ‘codetools’ not available?)

code for methods in class “Rcpp_SpatFactor” was not checked for suspicious field assignments (recommended package ‘codetools’ no

ERROR: Error in library(tidyterra): there is no package called ‘tidyterra’


In [2]:

# Configuration (do not containerize this cell)
param_minio_endpoint = "scruffy.lab.uvalight.net:9000"
param_minio_user_prefix = "zhanqing2016@gmail.com"  # Your personal folder in the naa-vre-user-data bucket in MinIO
secret_minio_access_key = "sFmE1jsm5hjJBBGh5RBL"
secret_minio_secret_key = "pczCG6FRpXQEtad7lAvXv00iCYFd5Dpa1g8GOWzR"
param_minio_region = "nl-uvalight"
library("aws.s3")
Sys.setenv("AWS_S3_ENDPOINT" = param_minio_endpoint,
           "AWS_DEFAULT_REGION" = param_minio_region,
           "AWS_ACCESS_KEY_ID" = secret_minio_access_key,
           "AWS_SECRET_ACCESS_KEY" = secret_minio_secret_key)

# List existing buckets: get a list of all available buckets
bucketlist()

# List files in bucket: get a list of files in a given bucket. For bucket `naa-vre-user-data`, only list files in your personal folder
get_bucket_df(bucket="naa-vre-user-data", prefix=paste0(param_minio_user_prefix, "/3D_Model"))

# Upload file to bucket: uploads `myfile_local.csv` to your personal folder on MinIO as `myfile.csv`
#put_object(bucket="naa-vre-user-data", file="myfile_local.csv", object=paste0(param_minio_user_prefix, "/myfile.csv"))

# Download file from bucket: download `myfile.csv` from your personal folder on MinIO and save it locally as `myfile_downloaded.csv`
#save_object(bucket="naa-vre-user-data", object=paste0(param_minio_user_prefix, "/myfile.csv"), file="myfile_downloaded.csv")


,Bucket,CreationDate
,<chr>,<chr>
1,naa-vre-public,2024-01-15T15:56:11.707Z
2,naa-vre-user-data,2024-02-14T16:46:55.908Z
3,naa-vre-waddenzee-shared,2024-05-15T12:22:45.112Z


,Key,LastModified,ETag,Size,Owner_ID,Owner_DisplayName,StorageClass,Bucket
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,zhanqing2016@gmail.com/3D_Model/sediment_mud_fraction.nc,2026-03-24T13:47:36.593Z,"""7f372bb94351a37a34574567357769ce-2""",6382520,02d6176db174dc93cb1b899f7c6078f08654445fe8cf1b6ce98d8855f66bdbf4,minio,STANDARD,naa-vre-user-data
2,zhanqing2016@gmail.com/3D_Model/topos2_dws_500m.nc,2026-03-24T13:44:04.757Z,"""ca5e7baa5afa5aadbf7e04cff4657434""",1286080,02d6176db174dc93cb1b899f7c6078f08654445fe8cf1b6ce98d8855f66bdbf4,minio,STANDARD,naa-vre-user-data


In [3]:
#ncfile_path <- "C:/Users/qzhan/OneDrive - NIOZ/Attachments/01_LTER-LIFE/03_Model/3D_models_WaddenSea/Input/"
ncfile_path <- "/home/jovyan/Cloud Storage/naa-vre-user-data/3D_Model/"
# --- Load topo (to get grid and Wadden island mask) ---
# nc_topo <- nc_open(paste0(ncfile_path, "topo_adjusted_dws_200m_2009.nc"))
nc_topo <- nc_open(paste0(ncfile_path, "topos2_dws_500m.nc"))
lonc <- ncvar_get(nc_topo, "lonc")
latc <- ncvar_get(nc_topo, "latc")
bathy <- ncvar_get(nc_topo, "bathymetry")
nc_close(nc_topo)

dim_xc <- dim(lonc)[1]
dim_yc <- dim(lonc)[2]

# Wadden island mask: TRUE = island (NA in topo)
island_mask <- is.na(bathy)

# --- Load Wadden Sea measurements ---
silt_nc <- terra::rast(paste0(ncfile_path, "sediment_mud_fraction.nc"))